# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import pandas as pd

from src import optuna_optimization as utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Ingest Feature Partitions and Environment Setup

In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [3]:
# Mapping Runtime Feature Schemas
nomod_columns = ['HasCrCard', 'IsActiveMember']
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

In [4]:
# Operational Parameters Constants
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

# Corrected Backing Store Paths
ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Establish SQLite tracking target link
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

## 2. Orchestrate and Initialize Search Parameter Studies

In [5]:
# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="average_precision",
    create_experiment=False,
    mlflow_kwargs={"nested": True}
)

study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0)
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_10429/2940032175.py:2: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-02 17:04:50,917] A new study created in memory with name: no-name-575fa85d-b968-40ab-8522-1636f7fca7f5


In [6]:
# Instantiate the cross-validation objective callable
objective_function = utils.ObjectiveCV(
    X=X_train, y=y_train, 
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns,
    n_splits=N_SPLITS, random_state=RANDOM_STATE
)

## 3. Run Optimization Search Workspace Execution Window

In [7]:
with mlflow.start_run(run_name="optuna_search"):
    
    study.optimize(
        objective_function,
        n_trials=500,
        callbacks=[mlflow_callback]
    )

    # Log global best performance summary attributes
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # Reconstruct and serialize top performing candidate artifacts pipeline
    best_pipeline = utils.build_pipeline(
        study.best_trial, 
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns,
        random_state=RANDOM_STATE
    )

    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns
    )

[I 2026-06-02 17:04:54,189] Trial 0 finished with value: 0.6803444513100079 and parameters: {'scaler': 'minmax', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 271, 'rf_max_depth': 17, 'rf_min_samples_split': 17, 'rf_min_samples_leaf': 4}. Best is trial 0 with value: 0.6803444513100079.
[I 2026-06-02 17:04:55,709] Trial 1 finished with value: 0.6995380376680137 and parameters: {'scaler': 'std', 'encoder': 'no_drop', 'model': 'xgb', 'xgb_n_estimators': 737, 'xgb_max_depth': 4, 'xgb_learning_rate': 0.05946518221249883, 'xgb_subsample': 0.9633503710020405, 'xgb_colsample_bytree': 0.6422484328764126, 'xgb_min_child_weight': 5, 'xgb_gamma': 3.778514323539329, 'xgb_reg_alpha': 0.0013603410602520518, 'xgb_reg_lambda': 2.3105855560764266e-06, 'xgb_scale_pos_weight': 2.4858268064321596}. Best is trial 1 with value: 0.6995380376680137.
[I 2026-06-02 17:04:57,583] Trial 2 finished with value: 0.6760651773372965 and parameters: {'scaler': 'std', 'encoder': 'no_drop', 'model': 'rf', 'rf

## 4. Display Session Optimization Diagnostics Results

In [8]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.7076

Optimal Parameter Combinations Selected:
 * scaler: minmax
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 825
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.060129937357623994
 * xgb_subsample: 0.9436299834068083
 * xgb_colsample_bytree: 0.6446229002292275
 * xgb_min_child_weight: 2
 * xgb_gamma: 4.987933640139174
 * xgb_reg_alpha: 0.0007778902988381235
 * xgb_reg_lambda: 2.206345157521368e-06
 * xgb_scale_pos_weight: 1.9338422010256977


Best average_precision: **0.7077109199922507**

Optimal Parameter Combinations Selected:
 * scaler: robust
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 588
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.06090819562411312
 * xgb_subsample: 0.967454071450258
 * xgb_colsample_bytree: 0.6262720546196823
 * xgb_min_child_weight: 3
 * xgb_gamma: 3.2784302986561005
 * xgb_reg_alpha: 0.0006480121274906255
 * xgb_reg_lambda: 7.441405323298544e-05
 * xgb_scale_pos_weight: 1.8090219464356987


Top Optimization Average Precision Score Target: **0.7079**

Optimal Parameter Combinations Selected:
 * scaler: minmax
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 967
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.05903303186469791
 * xgb_subsample: 0.9204856064383659
 * xgb_colsample_bytree: 0.6118872682736142
 * xgb_min_child_weight: 2
 * xgb_gamma: 4.2451076897824676
 * xgb_reg_alpha: 0.00019575646965291755
 * xgb_reg_lambda: 0.0005189779456101976
 * xgb_scale_pos_weight: 1.6516119336541517